# Neur — AS-RoPE vs RoPE vs Sinusoidal (Minimal A100 Pipeline)

**Goal:** Compare three positional encoding schemes on Hi→En NMT under a tight A100 budget (20–30h total).

**Design:**
- 1M deterministically-sampled pairs from the cleaned 5M Samanantar corpus
- Pre-tokenize once with IndicBART (Hi→En direction only), cache as flat int32 tensors
- 15K training steps per PE variant × 3 variants, effective batch = 256, bf16 + torch.compile + SDPA
- Fast greedy-only FLORES-200 eval (BLEU + chrF + TER) during the 3 runs
- One final full-metric pass (BLEU + chrF + TER + BERTScore + COMET + beam-5) on the winning PE variant

**Expected wallclock on A100:** ~3.5–4.5h total for all 3 runs + final eval. Leaves headroom for a second seed on the winner.

**Before you start:** Runtime → Change runtime type → GPU → **A100**.


## Step 0 — Mount Drive, clone repo, install deps

In [1]:
print("Hello NeurIPS ")

Hello NeurIPS 


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
%cd /content
!rm -rf /content/neur
!git clone https://github.com/thenileshmishra/AS-RoPE.git neur
%cd /content/neur
!git pull
!ls


/content
Cloning into 'neur'...
remote: Enumerating objects: 355, done.
remote: Counting objects: 100% (355/355), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 355 (delta 185), reused 278 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (355/355), 1.11 MiB | 10.46 MiB/s, done.
Resolving deltas: 100% (185/185), done.
/content/neur
Already up to date.
notebooks  pipeline  README.md	requirements.txt  src


In [4]:
!ls


notebooks  pipeline  README.md	requirements.txt  src


In [5]:
# Install minimal deps. Skips bert-score and unbabel-comet — those are only
# needed in step 4 (final eval on winner) and will be installed there.
!pip install -q transformers==4.44.2 sacrebleu tqdm sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 88.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.5 MB/s eta 0:00:00


In [6]:
import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())


PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics


In [7]:
# GPU sanity check — confirm A100 is attached
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'Memory: {props.total_memory / 1e9:.1f} GB')
    assert 'A100' in torch.cuda.get_device_name(0), 'Expected A100. Switch runtime type.'


CUDA: True
Device: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


## Step 1 — Download Samanantar + FLORES-200 (skip if already on Drive)

Skip this if you already have the raw data on Drive from a previous session.

In [ ]:
# Only run if you don't already have raw_data on Drive
!python -m pipeline.step1_download


## Step 2 — Clean and split (skip if already on Drive)

Skip this if `processed_data/train.tsv` already exists from a previous session.

In [ ]:
# Only run if processed_data is missing
!test -f /content/drive/MyDrive/neur/processed_data/train.tsv && echo 'processed_data exists, skipping' || python -m pipeline.step2_preprocess


In [ ]:
!ls -lh /content/drive/MyDrive/neur/processed_data/


## Step 2b — Sample 1M pairs + pre-tokenize (ONE-TIME, ~5–10 min)

Writes `train_1m_hi_en.pt` and `val_hi_en.pt` under `processed_data/tokenized/` on Drive. After this, training never touches the HuggingFace tokenizer.

In [7]:
!python -m pipeline.step2b_sample_and_tokenize \
  --n-train 1000000 \
  --max-seq-len 192 \
  --seed 42


[step2b] loading train pairs from /content/drive/MyDrive/neur/processed_data/train.tsv
[step2b] loaded 4,511,851 train pairs in 51.2s
[step2b] loading val pairs from /content/drive/MyDrive/neur/processed_data/val.tsv
[step2b] loaded 244,429 val pairs
[step2b] sampling 1,000,000 train pairs deterministically (seed=42)
[step2b] sanity check — first train pair:
         src = लोग अपने घरों से बाहर नही निकल रहे है।
         tgt = People were not moving out of their houses.
[step2b] building tokenizer: ai4bharat/IndicBART
tokenizer_config.json: 100% 498/498 [00:00<00:00, 3.59MB/s]
config.json: 100% 832/832 [00:00<00:00, 8.31MB/s]
spiece.model: 100% 1.90M/1.90M [00:01<00:00, 1.85MB/s]
added_tokens.json: 100% 221/221 [00:00<00:00, 1.83MB/s]
special_tokens_map.json: 100% 398/398 [00:00<00:00, 4.05MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavio

In [8]:
# Inspect what we produced — pay attention to seq_len_p99; if much less than 192,
# you can lower --max-seq-len in step 3 for a free speedup.
!cat /content/drive/MyDrive/neur/processed_data/tokenized/tokenization_summary.json


{
  "output_dir": "/content/drive/MyDrive/neur/processed_data/tokenized",
  "train_file": "/content/drive/MyDrive/neur/processed_data/tokenized/train_1m_hi_en.pt",
  "val_file": "/content/drive/MyDrive/neur/processed_data/tokenized/val_hi_en.pt",
  "train": {
    "n_examples": 1000000,
    "total_tokens": 50213009,
    "seq_len_mean": 50.213008880615234,
    "seq_len_p50": 41,
    "seq_len_p95": 114,
    "seq_len_p99": 170,
    "seq_len_max": 191
  },
  "val": {
    "n_examples": 244429,
    "total_tokens": 12306012,
    "seq_len_mean": 50.3459587097168,
    "seq_len_p50": 41,
    "seq_len_p95": 114,
    "seq_len_p99": 170,
    "seq_len_max": 191
  },
  "meta": {
    "tokenizer": "ai4bharat/IndicBART",
    "direction": "hi_en",
    "max_seq_len": 192,
    "seed": 42,
    "pad_id": 0,
    "sep_id": 64014,
    "eos_id": 3,
    "vocab_size": 64015
  }
}

## Step 2c — Copy tokenized files from Drive → local SSD

Drive I/O is 10–30× slower than Colab's local disk. Copy once, then train from `/content/`. This is the single biggest wall-clock win.

In [8]:
!mkdir -p /content/tokenized
!cp /content/drive/MyDrive/neur/processed_data/tokenized/train_1m_hi_en.pt /content/tokenized/
!cp /content/drive/MyDrive/neur/processed_data/tokenized/val_hi_en.pt /content/tokenized/
!ls -lh /content/tokenized/


total 487M
-rw------- 1 root root 391M Apr 13 18:16 train_1m_hi_en.pt
-rw------- 1 root root  96M Apr 13 18:16 val_hi_en.pt


## Step 3a — Train SINUSOIDAL baseline (~50–65 min on A100)

Uses: bf16, torch.compile, SDPA, pre-tokenized cached dataset, 500-sample in-loop eval, tqdm progress. All three runs use identical hyperparameters; only `--pe-type` differs.

In [13]:
!python -m pipeline.step3_train \
  --pe-type sinusoidal \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


[step3] using cached tokenized data: /content/tokenized/train_1m_hi_en.pt
[step3] device=cuda (NVIDIA A100-SXM4-40GB, 42.4 GB)
[step3] config: pe_type=sinusoidal d_model=256 n_layers=6 batch=64x4=256 steps=50000 max_seq_len=192 bf16=True compile=True cached=True
[train] loading cached tokenized train from /content/tokenized/train_1m_hi_en.pt
[train] loading cached tokenized val from /content/tokenized/val_hi_en.pt
[train] cached meta: train=1000000 val=244429 p99_len=170
[train] vocab_size=64015 pad_id=0 (from cached meta)
[train] capping in-loop val to 500 examples
[train] model params=21,120,768 pe_type=sinusoidal
[train] torch.compile enabled
[eval] step   5000 | val 7.2856 | ppl 1459.12 | tok/s 75099                                                     
[eval] step  10000 | val 5.3787 | ppl 216.74 | tok/s 77602                                                           
[eval] step  15000 | val 4.8041 | ppl 122.01 | tok/s 78867                                                         

## Step 3b — Train ROPE (~50–65 min on A100)

In [14]:
!python -m pipeline.step3_train \
  --pe-type rope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


[step3] using cached tokenized data: /content/tokenized/train_1m_hi_en.pt
[step3] device=cuda (NVIDIA A100-SXM4-40GB, 42.4 GB)
[step3] config: pe_type=rope d_model=256 n_layers=6 batch=64x4=256 steps=50000 max_seq_len=192 bf16=True compile=True cached=True
[train] loading cached tokenized train from /content/tokenized/train_1m_hi_en.pt
[train] loading cached tokenized val from /content/tokenized/val_hi_en.pt
[train] cached meta: train=1000000 val=244429 p99_len=170
[train] vocab_size=64015 pad_id=0 (from cached meta)
[train] capping in-loop val to 500 examples
[train] model params=21,120,768 pe_type=rope
[train] torch.compile enabled
train[rope]:   0% 0/50000 [00:00<?, ?step/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
[eval] step   5000 | val 7.1877 | ppl 1323.12 | tok/s 69541                                            

## Step 3c — Train AS-ROPE (~50–65 min on A100)

In [15]:
!python -m pipeline.step3_train \
  --pe-type asrope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


[step3] using cached tokenized data: /content/tokenized/train_1m_hi_en.pt
[step3] device=cuda (NVIDIA A100-SXM4-40GB, 42.4 GB)
[step3] config: pe_type=asrope d_model=256 n_layers=6 batch=64x4=256 steps=50000 max_seq_len=192 bf16=True compile=True cached=True
[train] loading cached tokenized train from /content/tokenized/train_1m_hi_en.pt
[train] loading cached tokenized val from /content/tokenized/val_hi_en.pt
[train] cached meta: train=1000000 val=244429 p99_len=170
[train] vocab_size=64015 pad_id=0 (from cached meta)
[train] capping in-loop val to 500 examples
[train] model params=21,121,536 pe_type=asrope
[train] torch.compile enabled
train[asrope]:   0% 0/50000 [00:00<?, ?step/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
[eval] step   5000 | val 7.1871 | ppl 1322.24 | tok/s 66240                                      

In [9]:
# Step 3d — train asrope2
!python -m pipeline.step3_train \
  --pe-type asrope2 \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --max-seq-len 192 \
  --seed 42


[step3] using cached tokenized data: /content/tokenized/train_1m_hi_en.pt
[step3] device=cuda (NVIDIA A100-SXM4-40GB, 42.4 GB)
[step3] config: pe_type=asrope2 d_model=256 n_layers=6 batch=64x4=256 steps=50000 max_seq_len=192 bf16=True compile=True cached=True
[train] loading cached tokenized train from /content/tokenized/train_1m_hi_en.pt
[train] loading cached tokenized val from /content/tokenized/val_hi_en.pt
[train] cached meta: train=1000000 val=244429 p99_len=170
[train] vocab_size=64015 pad_id=0 (from cached meta)
[train] capping in-loop val to 500 examples
[train] model params=21,122,304 pe_type=asrope2
[train] torch.compile enabled
train[asrope2]:   0% 0/50000 [00:00<?, ?step/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
[eval] step   5000 | val 7.1098 | ppl 1223.87 | tok/s 55378                                   

## Step 3e — Train AS-ROPE v3 (PA-RoPE: phase-adaptive)

AS-RoPE v2 + learnable per-(head, frequency) phase offsets `phi_q`, `phi_k`. At step 0, `phi=0` so it is identical to AS-RoPE v2. Adds only `2*n_heads*n_freqs` parameters.

In [ ]:
!python -m pipeline.step3_train \
  --pe-type asrope3 \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


## Step 3f — Train CC-ROPE (Content-Conditioned RoPE)

Frequency gates are computed per-token from a causal cumulative summary of the input embeddings. Projections are zero-initialized so gates start at 1.0 (= standard RoPE). Adds `2 * d_model * n_heads * n_freqs` parameters.

In [ ]:
!python -m pipeline.step3_train \
  --pe-type ccrope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


## Step 3g — Train DS-ROPE (Dual-Stream RoPE)

Blends two parallel position streams — absolute (`pos_abs`) and language-local (`pos_local` resets after `[SEP]`) — via a learnable per-(head, frequency) alpha. Heads can specialize: alpha→1 yields global attention, alpha→0 yields language-local attention. `sep_id` is read automatically from the cached tokenizer meta.

In [ ]:
!python -m pipeline.step3_train \
  --pe-type dsrope \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 50000 \
  --eval-every 5000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7


## Step 3d — Compare the three runs (fast metrics)

Reads the `metrics/<pe_type>/greedy/metrics.json` files and prints a side-by-side table.

In [ ]:
import json
from pathlib import Path

METRICS = Path('/content/drive/MyDrive/neur/outputs/metrics')
PE_TYPES = ('sinusoidal', 'rope', 'asrope', 'asrope2', 'asrope3', 'ccrope', 'dsrope')
rows = []
for pe in PE_TYPES:
    p = METRICS / pe / 'greedy' / 'metrics.json'
    if not p.exists():
        print(f'missing: {p}')
        continue
    m = json.loads(p.read_text())
    rows.append((pe, m.get('bleu', 0), m.get('chrf', 0), m.get('ter', 0)))

print(f"{'PE type':<14}{'BLEU':>10}{'chrF':>10}{'TER':>10}")
print('-' * 44)
for pe, bleu, chrf, ter in rows:
    print(f'{pe:<14}{bleu:>10.2f}{chrf:>10.2f}{ter:>10.2f}')

if rows:
    best = max(rows, key=lambda r: r[1])
    print(f'\nbest by BLEU: {best[0]} ({best[1]:.2f})')


In [ ]:
import json
from pathlib import Path

for pe in ['sinusoidal', 'rope', 'asrope', 'asrope2', 'asrope3', 'ccrope', 'dsrope']:
    p = Path(f'/content/drive/MyDrive/neur/outputs/metrics/{pe}/greedy/metrics.json')
    if not p.exists():
        print(f'{pe}: missing')
        continue
    m = json.loads(p.read_text())
    print(f"{pe}: rep_rate={m.get('repetition_rate','?'):.3f}  len_ratio={m.get('length_ratio','?'):.2f}")


In [ ]:
import json
from pathlib import Path
for pe in ['sinusoidal', 'rope', 'asrope', 'asrope2', 'asrope3', 'ccrope', 'dsrope']:
    log_path = Path(f'/content/drive/MyDrive/neur/outputs/logs/{pe}/metrics.jsonl')
    if not log_path.exists():
        print(f'\n{pe}: missing')
        continue
    rows = [json.loads(l) for l in log_path.read_text().splitlines() if l]
    evals = [(r['step'], r['val_loss']) for r in rows if 'val_loss' in r]
    print(f'\n{pe}:')
    for s, v in evals:
        print(f'  step {s:5d} | val_loss={v:.4f}')


In [13]:
import torch
ckpt = torch.load('/content/drive/MyDrive/neur/outputs/checkpoints/asrope2/best.pt', map_location='cpu')
gates = {k: v for k, v in ckpt['model_state_dict'].items() if 'freq_gates' in k}
for name, g in gates.items():
    print(f"{name}: mean={g.mean():.4f}  std={g.std():.4f}  min={g.min():.4f}  max={g.max():.4f}")


blocks.0.attn.freq_gates_q: mean=0.9046  std=0.0856  min=0.6460  max=1.0721
blocks.0.attn.freq_gates_k: mean=0.9792  std=0.1074  min=0.7516  max=1.2921
blocks.1.attn.freq_gates_q: mean=0.8786  std=0.1051  min=0.5999  max=1.1772
blocks.1.attn.freq_gates_k: mean=0.9964  std=0.1195  min=0.5942  max=1.3790
blocks.2.attn.freq_gates_q: mean=0.8847  std=0.1077  min=0.5678  max=1.0754
blocks.2.attn.freq_gates_k: mean=0.9995  std=0.1162  min=0.6817  max=1.4090
blocks.3.attn.freq_gates_q: mean=0.9375  std=0.0613  min=0.8045  max=1.2168
blocks.3.attn.freq_gates_k: mean=0.9558  std=0.0730  min=0.6508  max=1.1439
blocks.4.attn.freq_gates_q: mean=0.9277  std=0.0440  min=0.8190  max=1.0515
blocks.4.attn.freq_gates_k: mean=0.9719  std=0.0550  min=0.8379  max=1.1134
blocks.5.attn.freq_gates_q: mean=0.9173  std=0.0609  min=0.7703  max=1.0657
blocks.5.attn.freq_gates_k: mean=0.9859  std=0.0786  min=0.7781  max=1.2936


In [20]:
print("Hello Bro Done with Trainin check our results")

Hello Bro Done with Trainin check our results


## Step 4 — Full metric suite on the winning PE (~15–25 min)

Run this ONCE on the winning checkpoint to get BERTScore + COMET + beam-5 for the paper's headline table. Edit the `--checkpoint` path to match your winner.

In [14]:
# Install heavy metric deps only now — the 3 main runs skipped them
!pip install -q bert-score unbabel-comet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 121.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 53.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstat

In [ ]:
# EDIT the --checkpoint arg to point at your winner (sinusoidal / rope / asrope)
WINNER = 'asrope'  # change this to whichever PE type won
!python -m pipeline.step4_final_eval \
  --checkpoint /content/drive/MyDrive/neur/outputs/checkpoints/$WINNER/best.pt \
  --run-name ${WINNER}_final


## Step 5 — (Optional) Second seed on the winner for significance

If you have A100 hours left after the 3 main runs and the final eval, rerun the winning PE type with `--seed 7` to establish that the gain isn't a single-seed artifact. This is the single most valuable addition for paper reviewers.

In [ ]:
# Only run if you have spare budget. Edit WINNER below.
WINNER = 'asrope'
!python -m pipeline.step3_train \
  --pe-type $WINNER \
  --tokenized-train /content/tokenized/train_1m_hi_en.pt \
  --tokenized-val   /content/tokenized/val_hi_en.pt \
  --num-steps 15000 \
  --eval-every 3000 \
  --batch-size 64 --grad-accum 4 \
  --max-seq-len 192 \
  --seed 7 \
  --run-name ${WINNER}_seed7


## Step 6 — Verify outputs on Drive

In [ ]:
!echo '--- checkpoints ---'
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints/*/
!echo
!echo '--- logs ---'
!ls -lh /content/drive/MyDrive/neur/outputs/logs/*/
!echo
!echo '--- metrics ---'
!find /content/drive/MyDrive/neur/outputs/metrics -name '*.json' | head -20


In [23]:
import torch, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

ckpt = torch.load('/content/drive/MyDrive/neur/outputs/checkpoints/asrope/best.pt', map_location='cpu')
gates = {k: v.detach().numpy() for k, v in ckpt['model_state_dict'].items() if 'freq_gates' in k}

n_layers = len(gates)
gate_matrix = np.stack([gates[f'blocks.{i}.attn.freq_gates'] for i in range(n_layers)])  # [6, D/2]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap
im = axes[0].imshow(gate_matrix, aspect='auto', cmap='RdBu_r', vmin=0.5, vmax=1.5)
axes[0].set_xlabel('Frequency index (head_dim/2)')
axes[0].set_ylabel('Layer')
axes[0].set_yticks(range(n_layers))
axes[0].set_yticklabels([f'Layer {i}' for i in range(n_layers)])
axes[0].set_title('AS-RoPE Learned Frequency Gates')
plt.colorbar(im, ax=axes[0], label='Gate value (1.0 = standard RoPE)')

# Per-layer mean ± std
means = gate_matrix.mean(axis=1)
stds  = gate_matrix.std(axis=1)
axes[1].barh(range(n_layers), means, xerr=stds, align='center',
             color='steelblue', alpha=0.7, ecolor='black', capsize=4)
axes[1].axvline(x=1.0, color='red', linestyle='--', label='RoPE baseline (1.0)')
axes[1].set_yticks(range(n_layers))
axes[1].set_yticklabels([f'Layer {i}' for i in range(n_layers)])
axes[1].set_xlabel('Mean gate value ± std')
axes[1].set_title('Per-layer Spectral Adaptation')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/neur/outputs/asrope_gates.pdf', bbox_inches='tight', dpi=150)
plt.savefig('/content/drive/MyDrive/neur/outputs/asrope_gates.png', bbox_inches='tight', dpi=150)
print("Saved to Drive.")
plt.show()


Saved to Drive.


In [12]:

import sys, os
sys.path.insert(0, '/content/neur')
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
from pipeline import paths
from src.eval import compare_decoding

for pe in ['sinusoidal', 'rope', 'asrope']:
    ckpt = f'/content/drive/MyDrive/neur/outputs/checkpoints/{pe}/best.pt'
    out  = paths.METRICS_DIR / pe  # overwrites old metrics
    m = compare_decoding(ckpt, paths.RAW_FLORES, out, device='cuda', metrics_mode='fast')
    g = m['greedy']
    print(f"{pe}: BLEU={g['bleu']:.2f}  chrF={g['chrf']:.2f}  TER={g['ter']:.2f}  rep={g['repetition_rate']:.3f}")


[compare] running greedy decoding (metrics_mode=fast) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[eval] decoded 100/1012 (greedy)
[eval] decoded 200/1012 (greedy)
[eval] decoded 300/1012 (greedy)
[eval] decoded 400/1012 (greedy)
[eval] decoded 500/1012 (greedy)
[eval] decoded 600/1012 (greedy)
[eval] decoded 700/1012 (greedy)
[eval] decoded 800/1012 (greedy)
[eval] decoded 900/1012 (greedy)
[eval] decoded 1000/1012 (greedy)
═══════════════════════════════════
Evaluation Results
═══════════════════════════════════
Decoding            greedy
Num Samples         1012
───────────────────────────────────
BLEU                2.63
chrF                21.38
TER                 91.12
BERTScore-F1        N/A
COMET               N/A
Length Ratio        0.94
Repetition Rate     0.000
═══════════════════════════════════

Decoding                  BLEU          chrF           TER         COMET  BERTScore-F1
──────────────────────────────────────────────────────────────────────────────────────
greedy                    2.63         21.38         91.12           N/A           N/A
sinusoidal: BLEU

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[eval] decoded 100/1012 (greedy)
[eval] decoded 200/1012 (greedy)
[eval] decoded 300/1012 (greedy)
[eval] decoded 400/1012 (greedy)
[eval] decoded 500/1012 (greedy)
[eval] decoded 600/1012 (greedy)
[eval] decoded 700/1012 (greedy)
[eval] decoded 800/1012 (greedy)
[eval] decoded 900/1012 (greedy)
[eval] decoded 1000/1012 (greedy)
═══════════════════════════════════
Evaluation Results
═══════════════════════════════════
Decoding            greedy
Num Samples         1012
───────────────────────────────────
BLEU                3.75
chrF                26.65
TER                 100.88
BERTScore-F1        N/A
COMET               N/A
Length Ratio        1.14
Repetition Rate     0.000
═══════════════════════════════════

Decoding                  BLEU          chrF           TER         COMET  BERTScore-F1
──────────────────────────────────────────────────────────────────────────────────────
greedy                    3.75         26.65        100.88           N/A           N/A
rope: BLEU=3.75